In [8]:
import os
import folium
from folium.plugins import MarkerCluster
import pandas as pd


def convert_dms(deg, mn, sc, direction):
  """Convertit une coordonnée DMS en degrés décimaux."""
  d = float(deg) if pd.notna(deg) else 0.0
  m = float(mn) if pd.notna(mn) else 0.0
  s = float(sc) if pd.notna(sc) else 0.0
  dec = d + (m / 60.0) + (s / 3600.0)
  if str(direction).strip().upper() in ["S", "W"]:
    dec = -dec
  return dec


def get_color_from_ratio(ratio):
  """Dégradé simple du bleu (#3186cc) vers l'orange (#d95f02)."""
  if ratio < 0:
    ratio = 0.0
  if ratio > 1:
    ratio = 1.0
  r = int(49 + (217 - 49) * ratio)
  g = int(134 + (95 - 134) * ratio)
  b = int(204 + (2 - 204) * ratio)
  return f"#{r:02x}{g:02x}{b:02x}"


def add_legend(m, min_date_str, max_date_str):
  """Ajoute la légende en bas à gauche."""
  html = f"""
    <div style="
        position: fixed;
        bottom: 30px;
        left: 30px;
        z-index: 9999;
        background-color: white;
        padding: 10px 12px;
        border: 1px solid #ccc;
        border-radius: 6px;
        font-family: Arial, sans-serif;
        font-size: 12px;
        box-shadow: 0 2px 6px rgba(0,0,0,0.2);
    ">
        <div style="font-weight: bold; margin-bottom: 6px;">Mise en service 2026</div>
        <div style="
            width: 180px;
            height: 12px;
            background: linear-gradient(to right, #3186cc, #d95f02);
            border-radius: 3px;
            margin-bottom: 4px;
        "></div>
        <div style="display: flex; justify-content: space-between; font-size: 11px; color: #444;">
            <span>{min_date_str}</span>
            <span>{max_date_str}</span>
        </div>
    </div>
    """
  m.get_root().html.add_child(folium.Element(html))

def get_lieu(row):
  """Assemble l'adresse, le lieu-dit et le code postal."""
  morceaux = []
  for col in ["ADR_LB_ADD1", "ADR_LB_ADD2", "ADR_LB_ADD3", "ADR_LB_LIEU"]:
    val = str(row.get(col, "") or "").strip()
    if val and val.lower() != "nan" and val not in morceaux:
      morceaux.append(val)
  cp = str(row.get("ADR_NM_CP", "") or "").strip()
  if cp and cp.lower() != "nan":
    morceaux.append(cp)
  if morceaux:
    return " — ".join(morceaux)
  return "Lieu non renseigné"


DATA_DIR = r"C:\Users\bouyejay\Desktop\data\data"
SUPPORT_FILE = os.path.join(DATA_DIR, "sup_support.txt")
STATION_FILE = os.path.join(DATA_DIR, "sup_station.txt")
ANTENNE_FILE = os.path.join(DATA_DIR, "sup_antenne.txt")
EXPLOITANT_FILE = os.path.join(DATA_DIR, "SUP_EXPLOITANT.txt")

try:
  print("1. Chargement des fichiers ANFR...")
  df_support = pd.read_csv(
      SUPPORT_FILE, sep=";", encoding="latin-1", dtype={"STA_NM_ANFR": str}
  )
  df_station = pd.read_csv(
      STATION_FILE, sep=";", encoding="latin-1", dtype={"STA_NM_ANFR": str}
  )
  df_antenne = pd.read_csv(
      ANTENNE_FILE, sep=";", encoding="latin-1", dtype={"STA_NM_ANFR": str}
  )

  print("2. Conversion des coordonnées et filtre Métropole...")
  lats = []
  lons = []
  for _, r in df_support.iterrows():
    lats.append(
        convert_dms(
            r.get("COR_NB_DG_LAT"),
            r.get("COR_NB_MN_LAT"),
            r.get("COR_NB_SC_LAT"),
            r.get("COR_CD_NS_LAT"),
        )
    )
    lons.append(
        convert_dms(
            r.get("COR_NB_DG_LON"),
            r.get("COR_NB_MN_LON"),
            r.get("COR_NB_SC_LON"),
            r.get("COR_CD_EW_LON"),
        )
    )

  df_support["latitude"] = lats
  df_support["longitude"] = lons
  df_support = df_support[
      (df_support["longitude"] >= -5.5)
      & (df_support["longitude"] <= 10.0)
      & (df_support["latitude"] >= 41.0)
      & (df_support["latitude"] <= 51.5)
  ]

  print("3. Fusions, opérateur et filtre 2026...")
  df_support["SUP_ID"] = df_support["SUP_ID"].astype(str).str.strip()
  df_antenne["SUP_ID"] = df_antenne["SUP_ID"].astype(str).str.strip()
  df_support["STA_NM_ANFR"] = df_support["STA_NM_ANFR"].astype(str).str.strip()
  df_station["STA_NM_ANFR"] = df_station["STA_NM_ANFR"].astype(str).str.strip()

  df_merged = pd.merge(
      df_antenne,
      df_support.drop_duplicates("SUP_ID"),
      on=["SUP_ID", "STA_NM_ANFR"],
  )
  df_merged = pd.merge(df_merged, df_station, on="STA_NM_ANFR", how="left")

  if "ADM_ID" in df_merged.columns and os.path.exists(EXPLOITANT_FILE):
    df_exp = pd.read_csv(EXPLOITANT_FILE, sep=";", encoding="latin-1")
    df_exp["ADM_ID"] = df_exp["ADM_ID"].astype(str).str.strip()
    df_merged["ADM_ID"] = df_merged["ADM_ID"].astype(str).str.strip()
    df_merged = pd.merge(
        df_merged,
        df_exp[["ADM_ID", "ADM_LB_NOM"]].drop_duplicates("ADM_ID"),
        on="ADM_ID",
        how="left",
    )
    df_merged["Operateur"] = df_merged["ADM_LB_NOM"].fillna(df_merged["ADM_ID"])
  else:
    df_merged["Operateur"] = "Non renseigné"

  df_merged["Mise en service"] = pd.to_datetime(
      df_merged["DTE_EN_SERVICE"], format="%d/%m/%Y", errors="coerce"
  )
  df_2026 = (
      df_merged[df_merged["Mise en service"].dt.year == 2026]
      .sort_values(by="Mise en service", ascending=False)
      .head(2000)
  )

